# Benchmark: Split Flow

In [ ]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import cartopy.crs as ccrs
import subprocess
import os
import pandas as pd
import xarray as xr
import rasterio as rio
import glob

#### This benchmark is based on an experiment performed at the Syracuse University Lava Project that involved pouring molten basalt at a constant supply rate onto a sloping plane (Dietterich et al., [2015](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR20)).

This simulation captures both thermal effects and their impacts on flow rheology. This experiment also allows us to compare the flow propagation and surface temperature between the numerical simulations and the experimental data.

We encourage changing the volume betweens runs to compare the physical or stochastic reactions at various scales. It is also recommended to change the slope of the bed to measure various internal friction effects. Finally, the angle of the obstacle, which can dictate if/how the separated flows recombine.

If `custom_run` is set to `False`, a DEM that has been preformatted will be used, and the paramters will be ignored. If `True`, a new DEM will be dynamically generated using the provided parameters.

In [ ]:
custom_run = False

bed_slope = 14

volume = 7.7e-4

split_angle = 90

#### The following cells should not be edited, unless you have extensive knowledge of the model.

The following cell sets up common parameters. Following this, the next 4 cells format and generate input files for each respective model, at parameters considered reasonable for this benchmark.

In [ ]:
easting = 0
northing = 0
scale = volume/7.7e-4
dem = "DEMs_Lava_Benchmark/topography_dem_split.asc"
if custom_run:
    f = open("DEMs_Lava_Benchmark/slope.asc","r+")
    header = f.readlines()
    header[3] = f"""xllcorner {-2*scale}\n"""
    header[4] = f"""yllcorner {-1.5*scale}\n"""
    header[5] = f"""cellsize {0.01*scale}\n"""
    f.seek(0)
    f.writelines(header)
    f.truncate()
    f.close()
    
    r = rxr.open_rasterio("DEMs_Lava_Benchmark/slope.asc")
    height = r.rio.resolution()[0]*501*np.sin(np.deg2rad(bed_slope))
    slope = np.linspace(height,0,501)
    stacked = np.tile(slope,(301,1))
    r[0,:,:] = stacked
    y = 150
    x = 70
    for i in range(14):
        j = int(np.round(np.sin(np.deg2rad(split_angle))))
        r[0,y+j,x+j] = 5000
        r[0,y-j,x+j] = 5000
    r.rio.to_raster('Benchmark_Split.asc')
else:
    r = rxr.open_rasterio(dem)
xll, yll = int(r.x.min()), int(r.y.min())
coordinates = np.array([int(easting),int(northing)])
res = r.rio.resolution()[0]

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/Molasses')

f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = f"""MIN_RESIDUAL = {7e-6*scale}\n"""
conf[3] = f"""MAX_RESIDUAL ={1e-3*scale}\n"""
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/100}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = ../{dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()

subprocess.run("molasses custom_molasses.conf",shell=True, stderr=subprocess.DEVNULL)

victor.convert_molasses("molasses_split",resolution=res)

### MrLavaLoba/Flowy: Additional Notes

MrLavaLoba/Flowy (a drop in C++ replacement) input parameters are stored in `input.toml` in the `flowy` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/MrLavaLoba/')
g=open("input_data.py","r+")
inp = g.readlines()
inp[1] = """run_name = 'Split'\n"""
inp[3] = f"""source = '../{dem}'\n"""
inp[41] = """vent_flag = 0\n"""
inp[43] = f"""x_vent = [0]\n"""
inp[44] = f"""y_vent = [0]\n"""
inp[74] = f"""n_flows = 30\n"""
inp[79] = f"""min_n_lobes = 5000\n"""
inp[80] = f"""max_n_lobes = 5000\n"""
inp[98] = f"""lobe_area = .0002\n"""
inp[108] = f"""thickness_ratio = 1.25\n"""
inp[118] = f"""thickening_parameter = 0.75\n"""
inp[130] = f"""lobe_exponent = .03\n"""
inp[140] = f"""max_slope_prob = 0.92\n"""
inp[148] = f"""inertial_exponent = 0.4\n"""
inp[90] = f"""total_volume = {volume}\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("python mr_lava_loba.py",shell=True)

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/IMEX_LavaFlow')
radius = 2e-3/scale
subprocess.run("mv IMEX_LavaFlow_bm4.inp IMEX_LavaFlow.inp", shell=True)
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[100] = f""" VEL_SOURCE= {.4713*scale}\n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')
subprocess.run("mv IMEX_LavaFlow.inp IMEX_LavaFlow_bm4.inp", shell=True)

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/Lava2d/')

subprocess.run("python example_BM4_split.py", shell=True)
subprocess.run("python convert_output.py outputs/split/out.nc -o lava2d_split.asc", shell=True)

#### The final two cells concern visualization.

The first provides a reference to the final timestamp of each simulation, though all outputs are avilable in their models' respective folder. The second and last cell plots each model side by side. We encourage users to edit this to their personal preference.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

mll_outputs = glob.glob(f'mr_lava_loba/Split*thickness_masked*.asc')
mll_out = mll_outputs[-1]

r = rxr.open_rasterio("IMEX_LavaFlow/Split_max.asc",masked=True)
r2 = rxr.open_rasterio("Molasses/molasses_split.asc",masked=True)
r3 = rxr.open_rasterio(mll_out,masked=True)
r4 = rxr.open_rasterio("lava2d/lava2d_split.as",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min()/1000, r4.max()/1000

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2,subplot_kw=dict(projection=ccrs.epsg(32628)), figsize = (12,9))

xticks = np.linspace(-.2, 4.81, 5)
yticks = np.linspace(-1.5, 1.51, 5)

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))

thickness0 = victor.plot_flow(dem, "Molasses/molasses_split.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, mll_out, axes=ax1, zoom=False,title='MrLavaLoba',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, "IMEX_LavaFlow/Split_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow(dem, "Lava2d/lava2d_split.asc", axes=ax3 ,title='Lava2D', minimum=1e-10, scale=maximum*100)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")